# TimesFM zero-shot baseline

TimesFM is Google's pretrained time-series foundation model on Hugging Face. Like Chronos, it's a zero-shot forecaster — feed it a univariate context window of past prices, get a forecast back. **TimesFM is feature-poor by design** (it sees only the price series), so this is apples-to-oranges with the XGBoost baseline in `02_v2_models.ipynb`. Read it as: *what does a generic pretrained model think?*

In [10]:
import sys
print(sys.version)

3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


In [14]:
# Cell 1 - setup
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from models.utils import (
    chronological_split, select_enhanced_features,
    TARGET_REG, TARGET_CLASS,
    regression_report, classification_report,
    apply_feature_transforms,
)
FEATURES = PROJECT_ROOT / "data" / "features"


In [15]:
# Cell 2 - load & split
mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)
features = select_enhanced_features(mat)
print(f"{len(features)} features  train={len(train):,}  val={len(val):,}  test={len(test):,}")


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
87 features  train=60,429  val=8,760  test=8,767


In [16]:
# Install TimesFM

import timesfm
import torch

print("timesfm imported successfully")
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

timesfm imported successfully
torch version: 2.11.0+cu130
CUDA available: True


In [28]:
# Load pretrained TimesFM. Try 2.0 (500M, supports covariates) and fall back to 1.0 (200M).

device = "gpu" if torch.cuda.is_available() else "cpu"
torch_device = "cuda" if torch.cuda.is_available() else "cpu"  # torch uses "cuda" not "gpu"
print(f"timesfm version: 2.5-200m")
print(f"backend: {device}")
CONTEXT_LEN = 512
HORIZON = 1
try:
    config = timesfm.ForecastConfig(
        max_context=CONTEXT_LEN,
        max_horizon=HORIZON,
        per_core_batch_size=32,
    )
    tfm = timesfm.TimesFM_2p5_200M_torch(config=config)
    tfm.model = tfm.model.to(torch_device)  # move weights to GPU
    tfm.compile(forecast_config=config)
    TFM_VERSION = "2.5-200m"
except Exception as e:
    print(f"TimesFM 2.5 unavailable ({e})")
    tfm = None
    TFM_VERSION = "unavailable"
print(f"loaded TimesFM {TFM_VERSION}")

timesfm version: 2.5-200m
backend: gpu
loaded TimesFM 2.5-200m


In [29]:
# Build univariate test forecasts: for each 2024 timestamp, take prior 512h of HB_HUBAVG
# as context, forecast 1h ahead, store the median.
from tqdm.auto import tqdm

df_reg = mat.dropna(subset=[TARGET_REG]).sort_index()
_, _, te_reg = chronological_split(df_reg)
y_test = te_reg[TARGET_REG]

target_series = df_reg[TARGET_REG].astype("float32")
test_positions = target_series.index.get_indexer(y_test.index)
assert (test_positions >= CONTEXT_LEN).all(), "Need 512h of context for every test row"

BATCH = 64
preds = []
for start in tqdm(range(0, len(y_test), BATCH), desc="TimesFM test forecasts"):
    batch_pos = test_positions[start:start + BATCH]
    contexts = [
        target_series.iloc[p - CONTEXT_LEN:p].to_numpy()
        for p in batch_pos
    ]
    # 2.5 API: forecast(horizon, inputs) — freq argument removed
    point_forecast, _ = tfm.forecast(horizon=HORIZON, inputs=contexts)
    # point_forecast shape: (batch, horizon)
    preds.extend(point_forecast[:, 0].tolist())

timesfm_pred = pd.Series(preds, index=y_test.index, name=f"timesfm_{TFM_VERSION}")
print(timesfm_pred.head())
print(f"\nPredictions: {timesfm_pred.notna().sum():,} / {len(y_test):,}")

  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,428 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)


TimesFM test forecasts: 100%|██████████| 137/137 [00:13<00:00, 10.01it/s]

datetime
2024-01-01 00:00:00+00:00    21.459133
2024-01-01 01:00:00+00:00    20.879419
2024-01-01 02:00:00+00:00    19.308121
2024-01-01 03:00:00+00:00    20.822060
2024-01-01 04:00:00+00:00    21.965399
Name: timesfm_2.5-200m, dtype: float64

Predictions: 8,767 / 8,767


In [30]:
# Score on the 2024 test split.
report = regression_report(y_test.values, timesfm_pred.values, name=f"timesfm-{TFM_VERSION}-test")
report



--- Regression report: timesfm-2.5-200m-test ---
  MAE:             $32.72/MWh
  RMSE:            $138.07/MWh
  Spike recall:    2.63%
  Spike precision: 10.34%
---------------------------------



{'name': 'timesfm-2.5-200m-test',
 'mae': 32.72483138985001,
 'rmse': np.float64(138.0721575455505),
 'spike_recall': np.float64(0.02631578947368421),
 'spike_precision': np.float64(0.10344827586206896)}

## Optional: TimesFM 2.0 covariate path

TimesFM 2.0 supports exogenous covariates via `forecast_with_covariates`. If we successfully loaded 2.0 above, we additionally try the multivariate path with a small set of high-importance regressors. If only 1.0 is available, this cell is a no-op.

In [ ]:
# Cell - covariate path (gate on capability, not version string)
COVARIATES = ['hubavg_lag_1h', 'hubavg_lag_24h', 'hubavg_lag_168h',
              'temp_max_across_zones', 'wind_mean_across_zones',
              'load_actual_mw', 'henry_hub_price', 'da_HB_HUBAVG',
              'hour', 'is_peak_hours']

if tfm is None:
    print("TimesFM not loaded; skipping covariate path.")
elif hasattr(tfm, "forecast_with_covariates"):
    covariates = [c for c in COVARIATES if c in mat.columns]
    print(f"using {len(covariates)} covariates: {covariates}")
    SAMPLE_N = min(1000, len(y_test))
    sample_idx = y_test.index[:SAMPLE_N]
    sample_pos = test_positions[:SAMPLE_N]
    contexts = [target_series.iloc[p - CONTEXT_LEN:p].to_numpy() for p in sample_pos]
    dyn_num = {
        c: [mat[c].iloc[p - CONTEXT_LEN:p + HORIZON].to_numpy() for p in sample_pos]
        for c in covariates
    }
    try:
        cov_forecast, _ = tfm.forecast_with_covariates(
            inputs=contexts,
            dynamic_numerical_covariates=dyn_num,
            dynamic_categorical_covariates={},
            static_numerical_covariates={},
            static_categorical_covariates={},
        )
        cov_pred = pd.Series([f[0] for f in cov_forecast], index=sample_idx,
                             name=f"timesfm-{TFM_VERSION}-covariates")
        regression_report(
            y_test.loc[sample_idx].values, cov_pred.values,
            name=f"timesfm-{TFM_VERSION}-covariates-test (first 1000 rows)",
        )
    except TypeError as e:
        # API signature mismatch (likely on TimesFM 2.5 vs 2.0). Surface details.
        import inspect
        print(f"covariate API exists but signature mismatch: {type(e).__name__}: {e}")
        try:
            sig = inspect.signature(tfm.forecast_with_covariates)
            print(f"forecast_with_covariates signature: {sig}")
        except Exception:
            pass
        public = [m for m in dir(tfm) if not m.startswith("_")]
        print(f"tfm public methods/attrs: {public}")
    except Exception as e:
        print(f"covariate path failed: {type(e).__name__}: {e}")
else:
    print(f"Skipping covariate path: TimesFM {TFM_VERSION} does not expose forecast_with_covariates.")
    public = [m for m in dir(tfm) if not m.startswith("_")]
    print(f"tfm public methods/attrs: {public}")


## Local execution blocker

Authored but not executed end-to-end on the local `.venv` (Python 3.9). The `!pip install timesfm` cell does not pull in `torch` as a hard dep, and the local venv has no torch/CUDA stack — `import torch` raises `ModuleNotFoundError`. Additionally, TimesFM 2.0 weights are 500M parameters; downloading and running them needs the GPU server. **Run this notebook on the GPU Jupyter server**, where torch + CUDA are present and the HuggingFace cache can host the checkpoint.

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np
_y_true_dollar = np.asarray(y_test).reshape(-1)
_y_pred_dollar = np.asarray(timesfm_pred).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC: {_spike_pr_auc:.3f}")